<a href="https://colab.research.google.com/github/achmednasir/fooai/blob/main/fooai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Download fooai
%%capture
%cd /content
!git clone https://github.com/achmednasir/fooai.git

In [ ]:
#@title Prepare presets
%%capture
CIVITIA_API_KEY="" # @param {type:"string"}
MODELS = []
PRESET="GAME" # @param ["GAME", "GAME-HENTAI", "GAME-PONY"]

if PRESET == "GAME":
	MODELS = [
		(f"https://civitai.com/api/download/models/464939?type=Model&format=SafeTensor&token=" + CIVITIA_API_KEY, "models/checkpoints/PonyFaeTality.safetensors"),
		(f"https://civitai.com/api/download/models/297740?type=Model&format=SafeTensor&token=" + CIVITIA_API_KEY, "models/checkpoints/DynaVisionXL.safetensors"),
		(f"https://huggingface.co/Samirjahri/jug/resolve/main/Animated.safetensors?download=true", "models/loras/Animated.safetensors"),
		(f"https://huggingface.co/Samirjahri/jug/resolve/main/Pony3D.safetensors?download=true", "models/loras/Pony3D.safetensors"),
	]

In [ ]:
import requests
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
%cd /content/fooai

def get_content_length(url):
    try:
        response = requests.head(url)
        response.raise_for_status()
        return int(response.headers.get('content-length', 0))
    except requests.exceptions.RequestException as e:
        print(f"Request error for {url}: {e}")
    except Exception as e:
        print(f"Failed to get content length for {url}: {e}")
    return 0

def download_model(url, filename):
    try:
        total_size = get_content_length(url)
        if total_size == 0:
            raise ValueError(f"Failed to get content length for {url}")

        response = requests.get(url, stream=True)
        response.raise_for_status()
        chunk_size = 8192
        downloaded = 0

        with open(filename, 'wb') as file, tqdm(total=total_size, unit='iB', unit_scale=True, unit_divisor=1024, desc=filename) as bar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                file.write(chunk)
                downloaded += len(chunk)
                bar.update(len(chunk))

        return f"Downloaded {filename}"
    except requests.exceptions.RequestException as e:
        return f"Request error for {url}: {e}"
    except Exception as e:
        return f"Failed to download {filename} from {url}: {e}"

# Use ThreadPoolExecutor to manage concurrent downloads
with ThreadPoolExecutor(max_workers=len(MODELS)) as executor:
    futures = {executor.submit(download_model, url, filename): filename for url, filename in MODELS}

    for future in as_completed(futures):
        try:
            result = future.result()
            if result:
                print(result)
        except Exception as exc:
            print(f"An error occurred: {exc}")

In [ ]:
%cd /content/fooai
!python entry_with_update.py --preset empty --share --always-high-vram

In [ ]:
!zip -r /content/file.zip /content/fooai/outputs/

from google.colab import files
files.download("/content/file.zip")